# Lesson 15 — Capstone: an end-to-end computational-psychiatry project

This is where the course comes together. You will take a data set from the first look to a written conclusion, using every tool of the course in the order a real project uses them: describe the behaviour, fit candidate models, check that the comparison and the parameters can be trusted, relate a parameter to a symptom score without fooling yourself, test the model's predictions, and write it up. Then — because the data are simulated — you will see how close you came to the truth.

**The question.** *Is learning from positive feedback related to anhedonia?* The reinforcement-learning literature on depression suggests that reward-related learning is altered with anhedonia (Huys, Pizzagalli, Bogdan & Dayan, 2013), and the asymmetric-learning literature (lesson 13; Lefebvre et al., 2017) offers a precise way to phrase it: is the learning rate for positive prediction errors, $\alpha^+$, lower in people with higher anhedonia scores, while $\alpha^-$ is not?

**The data.** `bandit_anhedonia`: 80 simulated participants, 160 trials of a two-armed bandit whose better arm switches once, an anhedonia questionnaire score, an anxiety score (as a specificity control) and age. The data were generated by one of the course's models with a relation built in; which model and which relation is revealed at the end — do not peek at `datasets.truth` before milestone 8.

**Milestones**

1. Know your data (model-agnostic description)
2. Candidate models and fitting
3. Model recovery
4. Model comparison
5. Parameter recovery
6. The symptom analysis: effect, specificity, covariates, reliability
7. Posterior predictive check
8. Write-up, limitations, and the reveal

**Prerequisites:** lessons 1–14 (especially 5–7, 13 and 14).

**Estimated time:** 3–4 hours.

In [ ]:
import sys; sys.path.insert(0, "..")      # find the rlcourse package when running from notebooks/
import numpy as np
import matplotlib.pyplot as plt
from rlcourse import check, hint, reveal, progress, use, setup_style
from rlcourse import models, tasks, fitting, plotting
setup_style()
rng = np.random.default_rng(2024)        # one seed per notebook: results are reproducible

import pandas as pd
from scipy import stats
from rlcourse import datasets
from rlcourse.exercises.lesson14 import symptom_regression, split_half_reliability   # the tools you wrote in lesson 14

RUN_FULL = False     # True: refit every participant live (a few minutes); False: load the cached fits
import warnings
warnings.filterwarnings("ignore", message="overflow encountered in exp")   # runaway beta estimates overflow exp(); harmless

## Milestone 1 — Know your data

Never fit a model to data you have not looked at. Load the tables, read the task description, and plot the reward schedule.

In [ ]:
trials, subjects = datasets.load("bandit_anhedonia")
per_subject = datasets.split_by_subject(trials)         # {subject: {"choice": ..., "reward": ..., "p_arm0": ...}}
print(datasets.describe("bandit_anhedonia"))
print(trials.head())
print(subjects.describe().round(1))

p_arm0 = per_subject[0]["p_arm0"]                        # the same schedule for everyone
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(p_arm0, label="P(reward | arm 0)")
ax.plot(1 - p_arm0, label="P(reward | arm 1)")
ax.set(xlabel="trial", ylabel="reward probability", title="The task: a two-armed bandit with one reversal", ylim=(0, 1))
ax.legend()
plt.show()

### ✏️ Exercise 15.1 — A model-agnostic summary of one participant *(coding)*

Return a dictionary with four numbers for one participant: `reward_rate` (mean reward), `p_better` (proportion of choices of the currently better arm — arm 0 while `p_arm0 > 0.5`, arm 1 otherwise), `p_stay_win` (probability of repeating the previous choice after a reward) and `p_stay_lose` (after no reward).

In [ ]:
def summarise_subject(choices, rewards, p_arm0):
    """Model-agnostic summary: reward rate, P(better arm), P(stay | reward), P(stay | no reward)."""
    choices = np.asarray(choices)
    rewards = np.asarray(rewards, dtype=float)
    p_arm0 = np.asarray(p_arm0, dtype=float)
    better = ...                      # <- the better arm on each trial: 0 if p_arm0 > 0.5 else 1 (np.where)
    stay = ...                        # <- 1 if the choice on trial t repeats trial t-1 (length n-1)
    previous_reward = rewards[:-1]
    return {"reward_rate": float(np.mean(rewards)),
            "p_better": float(np.mean(choices == better)),
            "p_stay_win": ...,        # <- mean of stay where the previous reward was 1
            "p_stay_lose": ...}       # <- mean of stay where the previous reward was 0

check("15.1", summarise_subject)

In [ ]:
summarise_subject = use("15.1", summarise_subject)

summary = pd.DataFrame([dict(subject=s, **summarise_subject(d["choice"], d["reward"], d["p_arm0"])) for s, d in per_subject.items()])
summary = summary.merge(subjects, on="subject")
summary["anhedonia_tertile"] = pd.qcut(summary["anhedonia"].rank(method="first"), 3, labels=["low", "medium", "high"])
summary.groupby("anhedonia_tertile", observed=True)[["reward_rate", "p_better", "p_stay_win", "p_stay_lose"]].mean().round(3)

In [ ]:
# learning curves by anhedonia tertile: P(choose the better arm) on every trial
better_arm = np.where(p_arm0 > 0.5, 0, 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for tertile in ["low", "medium", "high"]:
    ids = summary.loc[summary["anhedonia_tertile"] == tertile, "subject"]
    curves = np.array([per_subject[s]["choice"] == better_arm for s in ids], dtype=float)
    smoothed = pd.Series(curves.mean(axis=0)).rolling(8, center=True, min_periods=1).mean()
    axes[0].plot(smoothed, label=f"{tertile} anhedonia (n = {len(ids)})")
axes[0].axvline(80, color="#52514e", ls="--", lw=1)
axes[0].set(xlabel="trial", ylabel="P(choose better arm)", title="Learning curves by anhedonia tertile (8-trial smoothing)", ylim=(0.3, 1))
axes[0].legend()
means = summary.groupby("anhedonia_tertile", observed=True)[["p_stay_win", "p_stay_lose"]].mean()
x = np.arange(3)
axes[1].bar(x - 0.18, means["p_stay_win"], 0.36, label="after reward")
axes[1].bar(x + 0.18, means["p_stay_lose"], 0.36, label="after no reward")
axes[1].set(xticks=x, xticklabels=means.index, ylabel="P(repeat previous choice)", title="Stay probabilities by anhedonia tertile", ylim=(0, 1))
axes[1].legend()
axes[1].grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

**Write down what you see** before moving on (a few bullet points in the cell below): does performance differ with anhedonia? Does the difference look like a difference in learning from rewards, from non-rewards, or in how deterministic choices are? These model-agnostic observations are what your model will later have to reproduce (milestone 7).

*Your observations:*

- …
- …

## Milestone 2 — Candidate models and fitting

A conclusion about $\alpha^+$ presupposes that an asymmetric learning model describes the data better than its rivals — or at least that we know how the rivals would have behaved. The course's five bandit models (`models.MODEL_SPECS`) are the candidate set:

| model | parameters | what it claims |
|---|---|---|
| `bias` | $b$ | no learning: a fixed preference for arm 0 |
| `wsls` | $\varepsilon$ | win-stay/lose-shift with lapses: a rule, no values |
| `rw` | $\alpha, \beta$ | Rescorla–Wagner values + softmax (lesson 3) |
| `rw_asym` | $\alpha^+, \alpha^-, \beta$ | separate learning rates for positive and negative prediction errors (lesson 13) |
| `rw_stick` | $\alpha, \beta, \kappa$ | RW + choice perseveration |

### ✏️ Exercise 15.2 — Fit all candidate models to one participant *(coding)*

Loop over `models.MODEL_SPECS` (name → `(nll_function, parameter_names, kinds)`), fit each model with `fitting.fit_mle` (data `(choices, rewards)`, the model's `kinds`, `n_starts` random starts, the given `rng`), and return `{name: {"nll": ..., "k": number of parameters, "params": {parameter: value}}}`.

In [ ]:
def fit_subject_all_models(choices, rewards, rng, n_starts=3):
    """Maximum-likelihood fits of every model in models.MODEL_SPECS to one participant."""
    out = {}
    for name, (nll_fn, param_names, kinds) in models.MODEL_SPECS.items():
        raise NotImplementedError("fit this model and store nll, k and params under out[name]")
    return out

check("15.2", fit_subject_all_models)

In [ ]:
fit_subject_all_models = use("15.2", fit_subject_all_models)

for s in [0, 1, 2]:                                       # three live fits, to see the procedure
    fits = fit_subject_all_models(per_subject[s]["choice"], per_subject[s]["reward"].astype(float), np.random.default_rng(100 + s))
    print(f"subject {s}: " + ", ".join(f"{m}: nll {v['nll']:.1f}" for m, v in fits.items()))

if RUN_FULL:
    rows = []
    for s, d in per_subject.items():
        fits = fit_subject_all_models(d["choice"], d["reward"].astype(float), np.random.default_rng(100 + s), n_starts=5)
        for m, v in fits.items():
            rows.append(dict(subject=s, model=m, k=v["k"], n=len(d["choice"]), nll=v["nll"], **v["params"]))
    all_fits = pd.DataFrame(rows)
    all_fits["bic"] = fitting.bic(all_fits["nll"], all_fits["k"], all_fits["n"])
else:
    all_fits = pd.read_csv("../data/bandit_anhedonia_fits.csv")      # produced by scripts/make_fits.py (5 random starts)
all_fits["aic"] = fitting.aic(all_fits["nll"], all_fits["k"])
print(all_fits.head(10).round(3))

## Milestone 3 — Model recovery

Before trusting the comparison, ask what it would say if each model were true. We simulate participants from each candidate — with parameters resampled from the fitted values, so that the simulations look like our sample — fit all five models to each, and tabulate which model wins. The simulator below runs any of the five models on the task's actual reward schedule (the same loop as `models.simulate_model`, with the reversal).

In [ ]:
def simulate_on_schedule(name, params, p_arm0, rng, q0=0.5):
    """Simulate one of the five candidate models on the capstone reward schedule."""
    n = len(p_arm0)
    Q = np.full(2, q0)
    choices, rewards = np.zeros(n, int), np.zeros(n)
    prev_c = prev_r = None
    for t in range(n):
        if name == "bias":
            p = np.array([params[0], 1 - params[0]])
        elif name == "wsls":
            eps = params[0]
            if prev_c is None:
                p = np.array([0.5, 0.5])
            else:
                p = np.full(2, eps / 2)
                p[prev_c if prev_r > 0 else 1 - prev_c] += 1 - eps
        elif name == "rw_stick":
            alpha, beta, kappa = params
            bonus = np.zeros(2)
            if prev_c is not None:
                bonus[prev_c] = kappa
            p = models.softmax(beta * Q + bonus, 1.0)
        else:                                               # rw, rw_asym: softmax on values
            p = models.softmax(Q, params[-1])
        c = models.choose(p, rng)
        r = float(rng.random() < (p_arm0[t] if c == 0 else 1 - p_arm0[t]))
        if name in ("rw", "rw_stick"):
            Q[c] += params[0] * (r - Q[c])
        elif name == "rw_asym":
            delta = r - Q[c]
            Q[c] += (params[0] if delta > 0 else params[1]) * delta
        choices[t], rewards[t] = c, r
        prev_c, prev_r = c, r
    return choices, rewards

def sample_fitted_params(name, rng, beta_cap=20.0):
    """Draw a parameter vector from the fitted values of one model (one random participant),
    capping runaway inverse temperatures."""
    names = models.MODEL_SPECS[name][1]
    row = all_fits[all_fits["model"] == name].sample(1, random_state=int(rng.integers(1e9))).iloc[0]
    params = [min(row[p], beta_cap) if p == "beta" else row[p] for p in names]
    return params

model_names = list(models.MODEL_SPECS)
n_sims = 12
best_bic = np.zeros((5, 5)); best_aic = np.zeros((5, 5))
for i, true_model in enumerate(model_names):
    for _ in range(n_sims):
        params = sample_fitted_params(true_model, rng)
        choices, rewards = simulate_on_schedule(true_model, params, p_arm0, rng)
        fits = fit_subject_all_models(choices, rewards, rng, n_starts=2)
        bics = [fitting.bic(fits[m]["nll"], fits[m]["k"], len(choices)) for m in model_names]
        aics = [fitting.aic(fits[m]["nll"], fits[m]["k"]) for m in model_names]
        best_bic[i, int(np.argmin(bics))] += 1
        best_aic[i, int(np.argmin(aics))] += 1
confusion_bic, confusion_aic = best_bic / n_sims, best_aic / n_sims

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
plotting.plot_confusion(confusion_bic, model_names, ax=axes[0], title="Model recovery with BIC\n(rows: simulated model, cols: winner)")
plotting.plot_confusion(confusion_aic, model_names, ax=axes[1], title="Model recovery with AIC\n(rows: simulated model, cols: winner)")
plt.tight_layout()
plt.show()

Look at the `rw_asym` row. With 160 trials and the modest asymmetries typical of our sample, BIC's penalty ($\ln 160 \approx 5.1$ per parameter) almost never lets the asymmetric model win *even when it generated the data*; AIC (penalty 2) detects it somewhat more often, but at the cost of frequently preferring `rw_asym` for data that plain RW generated. Neither criterion can adjudicate between RW and asymmetric RW in this design. This is the single most important fact for interpreting milestone 4: **a win for `rw` is not evidence against asymmetry** (and under AIC a win for `rw_asym` is only weak evidence for it). The comparison is informative about the other models — the no-learning and rule-based models are recovered well and lose clearly.

### ✏️ Exercise 15.3 — Reading the model-recovery matrix *(conceptual)*

In your model recovery, data simulated from the asymmetric model (`rw_asym`) are best fit by the plain RW model in a large share of the simulations, while data simulated from RW are almost never best fit by `rw_asym` under BIC. What does this mean?

**(a)** The asymmetry is hard to detect with 160 trials: when the simulated $\alpha^+$ and $\alpha^-$ are similar the simpler model wins through the BIC penalty, so evidence *for* `rw_asym` in real data is conservative and a win for RW is ambiguous.
**(b)** The asymmetric model is wrong and should be dropped from the comparison.
**(c)** BIC is biased and should be replaced by the raw log-likelihood.
**(d)** The simulations used too many trials.

In [ ]:
check("15.3", None)   # <- replace None with "a", "b", "c" or "d"

## Milestone 4 — Model comparison

Now the real participants. We report, for BIC and AIC: the summed criterion across participants (fixed-effects comparison), how many participants each model wins, and the random-effects Bayesian model selection of lesson 7 (`fitting.bms`, with $-\text{BIC}/2$ or $-\text{AIC}/2$ as the per-participant log evidence): expected frequencies and protected exceedance probabilities.

In [ ]:
subject_ids = sorted(all_fits["subject"].unique())
def evidence_matrix(criterion):
    return np.array([[-all_fits[(all_fits.subject == s) & (all_fits.model == m)][criterion].iloc[0] / 2 for m in model_names]
                     for s in subject_ids])

table = []
for criterion in ["bic", "aic"]:
    L = evidence_matrix(criterion)
    res = fitting.bms(L, rng=rng)
    summed = all_fits.groupby("model")[criterion].sum()
    wins = all_fits.loc[all_fits.groupby("subject")[criterion].idxmin(), "model"].value_counts()
    for j, m in enumerate(model_names):
        table.append({"criterion": criterion.upper(), "model": m, "summed": round(summed[m], 1), "participants won": int(wins.get(m, 0)),
                      "expected frequency": round(res["r"][j], 2), "PXP": round(res["pxp"][j], 2)})
pd.DataFrame(table).set_index(["criterion", "model"])

Learning models beat the no-learning and rule-based models decisively, but the two criteria disagree about asymmetry, and milestone 3 told us why: in this design BIC has almost no power to detect it and AIC trades a few detections for many false alarms. Formally, the comparison is **inconclusive about asymmetry** — which is not the same as "there is no asymmetry". Our hypothesis is specifically about $\alpha^+$, so we continue with `rw_asym` as the *hypothesis model*, report the comparison honestly, and later check whether the conclusion depends on this choice (milestone 6, robustness). A better design — more trials, or outcomes that make positive and negative prediction errors equally frequent — would be the real fix.

### ✏️ Exercise 15.4 — A BIC difference by hand *(maths)*

For one participant with $n = 160$ trials, the RW model ($k = 2$) reaches $\text{nll} = 95.2$ and the asymmetric model ($k = 3$) $\text{nll} = 92.1$. Compute $\text{BIC}(\text{rw}) - \text{BIC}(\text{rw\_asym})$ (positive values favour the asymmetric model). Use the natural logarithm.

In [ ]:
answer = None   # <- your number
check("15.4", answer)

## Milestone 5 — Parameter recovery

Can this design recover $\alpha^+$, $\alpha^-$ and $\beta$ from 160 trials, and do they trade off? Simulate 30 participants from the asymmetric model with parameters resampled from the fits, refit, and compare.

In [ ]:
true_params, recovered = [], []
for i in range(30):
    params = sample_fitted_params("rw_asym", rng)
    choices, rewards = simulate_on_schedule("rw_asym", params, p_arm0, rng)
    fit = fitting.fit_mle(models.nll_rw_asym, (choices, rewards), ["unit", "unit", "pos"], n_starts=3, rng=rng)
    true_params.append(params)
    recovered.append(fit["params"])
true_params, recovered = np.array(true_params), np.array(recovered)
recovered[:, 2] = np.minimum(recovered[:, 2], 50)            # cap runaway β for the plot

plotting.plot_parameter_recovery(np.c_[true_params[:, :2], np.log(true_params[:, 2])],
                                 np.c_[recovered[:, :2], np.log(recovered[:, 2])], ["α⁺", "α⁻", "log β"])
plt.show()
print("correlations among recovered parameters (trade-off check):")
print(pd.DataFrame(np.c_[recovered[:, :2], np.log(recovered[:, 2])], columns=["α⁺", "α⁻", "log β"]).corr().round(2))
print("correlations among the *generating* parameters:")
print(pd.DataFrame(np.c_[true_params[:, :2], np.log(true_params[:, 2])], columns=["α⁺", "α⁻", "log β"]).corr().round(2))

$\alpha^+$ recovers reasonably, $\alpha^-$ less well, and the recovered $\alpha^+$ and $\log\beta$ are clearly negatively correlated — more so than the generating values. That excess is the estimation trade-off of lesson 14: fast learning from rewards and deterministic choice produce similar data. For the symptom analysis this means two things: report the relation of *every* parameter to the symptom (if anhedonia related to $\beta$ as well, the $\alpha^+$ result would be suspect), and prefer regularised or hierarchical estimates when you can.

### ✏️ Exercise 15.5 — Interpreting parameter recovery before the symptom analysis *(conceptual)*

Suppose that in this design $\alpha^+$ recovers with $r = 0.9$ but $\alpha^-$ only with $r = 0.4$ (and recovered $\alpha^-$ correlates with recovered $\beta$). What follows for the symptom analysis?

**(a)** A null result for $\alpha^-$ would be uninformative and a positive one hard to interpret; conclusions about negative-feedback learning need a better design (more trials or more negative prediction errors) or hierarchical estimation.
**(b)** $\alpha^-$ can be dropped from the model because it does not exist.
**(c)** The fitting code must be wrong, since both learning rates are estimated by the same procedure.
**(d)** $r = 0.4$ is adequate for correlating $\alpha^-$ with symptoms as long as the sample is large.

In [ ]:
check("15.5", None)   # <- replace None with "a", "b", "c" or "d"

## Milestone 6 — The symptom analysis

Three questions, in order: *Is there an effect?* (correlation with a confidence interval), *Is it specific?* (other parameters, the other symptom score, covariates), and *How much can we trust the measurement?* (reliability, attenuation). The tools are the ones you built in lesson 14.

In [ ]:
asym = all_fits[all_fits["model"] == "rw_asym"].merge(subjects, on="subject").sort_values("subject").reset_index(drop=True)
asym["log_beta"] = np.log(asym["beta"])

def correlation_with_ci(x, y):
    r, p = stats.pearsonr(x, y)
    z, se = np.arctanh(r), 1 / np.sqrt(len(x) - 3)
    return r, p, np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)

r, p, lo, hi = correlation_with_ci(asym["alpha_pos"], asym["anhedonia"])
rho, p_rho = stats.spearmanr(asym["alpha_pos"], asym["anhedonia"])
print(f"alpha+ ~ anhedonia: r = {r:.2f} [95% CI {lo:.2f}, {hi:.2f}], p = {p:.4f};  Spearman rho = {rho:.2f}")

table = []
for param in ["alpha_pos", "alpha_neg", "log_beta"]:
    for symptom in ["anhedonia", "anxiety"]:
        b, pv = symptom_regression(asym, param, symptom, covariates=("age",))
        table.append({"parameter": param, "symptom": symptom, "std. beta (age-controlled)": round(b, 2), "p": round(pv, 4)})
display(pd.DataFrame(table).pivot(index="parameter", columns="symptom"))

fig, ax = plt.subplots(figsize=(5, 3.8))
ax.scatter(asym["anhedonia"], asym["alpha_pos"], s=24, alpha=0.8, edgecolor="white")
slope, intercept = np.polyfit(asym["anhedonia"], asym["alpha_pos"], 1)
xx = np.linspace(asym["anhedonia"].min(), asym["anhedonia"].max(), 50)
ax.plot(xx, intercept + slope * xx, color="#52514e", lw=1.5)
ax.set(xlabel="anhedonia score", ylabel="fitted α⁺ (rw_asym)", title=f"Positive learning rate and anhedonia: r = {r:.2f}")
plt.show()

**Robustness to the model choice.** Milestone 4 left the asymmetric model's status open. If the effect is real, it should also show — in diluted form — in the single learning rate of the plain RW model, which averages over positive and negative errors. Check it:

In [ ]:
rw = all_fits[all_fits["model"] == "rw"].merge(subjects, on="subject")
b, pv = symptom_regression(rw, "alpha", "anhedonia", covariates=("age",))
print(f"RW model, alpha ~ anhedonia (age-controlled): standardised beta = {b:.2f}, p = {pv:.4f}")

**Reliability and attenuation.** Estimate the split-half reliability of $\alpha^+$ with the better method of lesson 14 — interleaved blocks of 10 trials, the full sequence replayed, a weak prior — and, for contrast, the naive first-half/second-half maximum-likelihood version. Then correct the observed correlation for attenuation, treating the questionnaire's reliability as 0.9 (a typical published value for a well-constructed scale).

In [ ]:
def nll_rw_asym_masked(params, choices, rewards, mask, q0=0.5):
    """Asymmetric-RW likelihood scoring only the trials where mask is True (full history replayed)."""
    alpha_pos, alpha_neg, beta = params
    Q = np.full(2, q0)
    nll = 0.0
    for t, (c, r_t) in enumerate(zip(choices, rewards)):
        p = models.softmax(Q, beta)
        if mask[t]:
            nll -= np.log(p[c])
        delta = r_t - Q[c]
        Q[c] = Q[c] + (alpha_pos if delta > 0 else alpha_neg) * delta
    return nll

kinds_asym = ["unit", "unit", "pos"]
weak_prior = fitting.normal_log_prior(mu=[0.0, 0.0, 1.0], sd=[1.5, 1.5, 1.0])
block = np.arange(160) // 10
good = [[], []]; naive = [[], []]
for s, d in per_subject.items():
    choices, rewards = d["choice"], d["reward"].astype(float)
    for h, mask in enumerate([block % 2 == 0, block % 2 == 1]):
        fit = fitting.fit_mle(nll_rw_asym_masked, (choices, rewards, mask), kinds_asym, n_starts=2,
                              rng=np.random.default_rng(5000 + s + 100000 * h), log_prior=weak_prior)
        good[h].append(fit["params"])
    for h, sl in enumerate([slice(0, 80), slice(80, 160)]):
        fit = fitting.fit_mle(models.nll_rw_asym, (choices[sl], rewards[sl]), kinds_asym, n_starts=2,
                              rng=np.random.default_rng(5500 + s + 100000 * h))
        naive[h].append(fit["params"])
good, naive = [np.array(g) for g in good], [np.array(v) for v in naive]
rel_pos = split_half_reliability(good[0][:, 0], good[1][:, 0])
rel_neg = split_half_reliability(good[0][:, 1], good[1][:, 1])
rel_pos_naive = split_half_reliability(naive[0][:, 0], naive[1][:, 0])
rel_questionnaire = 0.9
r_corrected = r / np.sqrt(rel_pos * rel_questionnaire)
print(f"split-half reliability (interleaved blocks, weak prior): alpha+ = {rel_pos:.2f}, alpha- = {rel_neg:.2f}")
print(f"split-half reliability (naive halves, MLE):              alpha+ = {rel_pos_naive:.2f}")
print(f"observed r = {r:.2f}; corrected for attenuation (rel = {rel_pos:.2f} x {rel_questionnaire}): {r_corrected:.2f}")
print(f"ceiling on the observable correlation: {np.sqrt(rel_pos * rel_questionnaire):.2f}")
if abs(r_corrected) > 1:
    print("(a corrected value beyond ±1 means the reliability estimate is too low to support the correction)")

### ✏️ Exercise 15.6 — Correcting a correlation for attenuation *(maths)*

An observed correlation of $r = -0.30$ between a parameter with reliability 0.6 and a questionnaire with reliability 0.8: what is the attenuation-corrected estimate of the true correlation?

In [ ]:
answer = None   # <- your number
check("15.6", answer)

## Milestone 7 — Posterior predictive check

A model that fits is not necessarily a model that *behaves* like the participants. Simulate each participant from their fitted asymmetric-model parameters and compare the model-agnostic summaries of milestone 1 — the learning curves by anhedonia tertile and the stay probabilities — between real and simulated data.

### ✏️ Exercise 15.7 — Simulate a participant from the fitted asymmetric model *(coding)*

Write the simulator of the asymmetric model on the capstone schedule: `p_arm0[t]` is the reward probability of arm 0 on trial $t$ (arm 1 pays with $1 - $ `p_arm0[t]`). Return `{"choices", "rewards", "Q"}` with `Q` the values *before* each choice. The checker verifies the internal consistency of your simulation (as in exercise 3.8) and that the agent learns.

In [ ]:
def simulate_asym_subject(alpha_pos, alpha_neg, beta, p_arm0, rng, q0=0.5):
    """RW learner with asymmetric learning rates on the capstone task."""
    p_arm0 = np.asarray(p_arm0, dtype=float)
    n_trials = len(p_arm0)
    Q = np.full(2, q0, dtype=float)
    choices = np.zeros(n_trials, dtype=int)
    rewards = np.zeros(n_trials)
    Q_history = np.zeros((n_trials, 2))
    for t in range(n_trials):
        raise NotImplementedError("record Q, choose (softmax), sample the reward, update with alpha_pos or alpha_neg")
    return {"choices": choices, "rewards": rewards, "Q": Q_history}

check("15.7", simulate_asym_subject)

In [ ]:
simulate_asym_subject = use("15.7", simulate_asym_subject)

sim_summary = []
sim_curves = {}
for _, row in asym.iterrows():
    sim = simulate_asym_subject(row["alpha_pos"], row["alpha_neg"], min(row["beta"], 50), p_arm0, rng)
    sim_summary.append(dict(subject=int(row["subject"]), **summarise_subject(sim["choices"], sim["rewards"], p_arm0)))
    sim_curves[int(row["subject"])] = sim["choices"] == better_arm
sim_summary = pd.DataFrame(sim_summary).merge(summary[["subject", "anhedonia_tertile"]], on="subject")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for tertile, color in zip(["low", "high"], [plotting.PALETTE[0], plotting.PALETTE[1]]):
    ids = summary.loc[summary["anhedonia_tertile"] == tertile, "subject"]
    real = pd.Series(np.mean([per_subject[s]["choice"] == better_arm for s in ids], axis=0)).rolling(8, center=True, min_periods=1).mean()
    simd = pd.Series(np.mean([sim_curves[s] for s in ids], axis=0)).rolling(8, center=True, min_periods=1).mean()
    axes[0].plot(real, color=color, label=f"{tertile} anhedonia: data")
    axes[0].plot(simd, color=color, ls="--", label=f"{tertile} anhedonia: model")
axes[0].axvline(80, color="#52514e", ls="--", lw=1)
axes[0].set(xlabel="trial", ylabel="P(choose better arm)", title="Posterior predictive check: learning curves", ylim=(0.3, 1))
axes[0].legend(fontsize=8)
for key, marker, label in [("p_stay_win", "o", "P(stay | reward)"), ("p_stay_lose", "s", "P(stay | no reward)")]:
    axes[1].scatter(summary.sort_values("subject")[key], sim_summary.sort_values("subject")[key], s=18, alpha=0.7, marker=marker, label=label)
axes[1].plot([0, 1], [0, 1], ls="--", color="#52514e", lw=1)
axes[1].set(xlabel="data", ylabel="model simulation", title="Posterior predictive check: stay probabilities", xlim=(0, 1), ylim=(0, 1))
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

If the dashed and solid curves overlap and the points lie near the diagonal, the fitted model reproduces the behaviour it was fitted to — including the group difference. Systematic deviations (for example, simulated participants staying less after a loss than real ones) would point to a missing process, and would be the first thing to mention in the write-up.

## Milestone 8 — Write-up, limitations, and the reveal

Fill in the template below (a few sentences per heading). Numbers come from your cells above.

### Report: learning from positive feedback and anhedonia (simulated data)

**Methods.** *Task and sample:* … *Candidate models:* … *Fitting:* maximum likelihood with … random starts; *model comparison:* …; *validation:* model recovery (…), parameter recovery (…); *symptom analysis:* …

**Results.** *Behaviour:* … *Model comparison:* … *Parameter–symptom relation:* $\alpha^+$ ~ anhedonia r = …, 95% CI …, age-controlled standardised β = …; specificity: $\alpha^-$ …, anxiety …; robustness to model choice: … *Reliability:* split-half of $\alpha^+$ = …, attenuation-corrected r = … *Posterior predictive check:* …

**Limitations.** (exercise 15.8)

**Conclusion.** …

### ✏️ Exercise 15.8 — Limitations *(conceptual, written)*

Write the Limitations paragraph of your report (3–6 sentences). The checker looks for the main themes and then shows a model paragraph.

In [ ]:
check("15.8", """
    Write your limitations paragraph here.
""")

### The reveal

The data were generated by the asymmetric model, with anhedonia built into $\alpha^+$ only. Compare your estimates with the truth.

In [ ]:
truth = datasets.truth("bandit_anhedonia")
print("generating model: rw_asym (alpha_pos, alpha_neg, beta); anhedonia was built into alpha_pos only")
r_true, *_ = correlation_with_ci(truth["alpha_pos"], subjects["anhedonia"])
r_true_neg, *_ = correlation_with_ci(truth["alpha_neg"], subjects["anhedonia"])
print(f"true alpha+ ~ anhedonia: r = {r_true:.2f}    (observed with fitted alpha+: {r:.2f}; attenuation-corrected: {r_corrected:.2f})")
print(f"true alpha- ~ anhedonia: r = {r_true_neg:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, name, fitted in zip(axes, ["alpha_pos", "alpha_neg", "beta"], [asym["alpha_pos"], asym["alpha_neg"], np.log(asym["beta"])]):
    true_vals = np.log(truth[name]) if name == "beta" else truth[name]
    rr = np.corrcoef(true_vals, fitted)[0, 1]
    ax.scatter(true_vals, fitted, s=20, alpha=0.8, edgecolor="white")
    lo_, hi_ = min(true_vals.min(), fitted.min()), max(true_vals.max(), fitted.max())
    ax.plot([lo_, hi_], [lo_, hi_], ls="--", color="#52514e", lw=1)
    label = "log β" if name == "beta" else name
    ax.set(xlabel=f"true {label}", ylabel=f"fitted {label}", title=f"{label}: r = {rr:.2f}")
plt.tight_layout()
plt.show()

Compare three numbers: the *true* correlation between $\alpha^+$ and anhedonia, the *observed* one, and your *attenuation-corrected* estimate. The gap between the first two is entirely estimation noise in $\alpha^+$ — the attenuation of lesson 14 in action — and the corrected estimate should land closer to the truth. Also note how the formal model comparison (milestone 4) looked when the asymmetric model was in fact the true one: the design, not the model, was the limiting factor.

### Extensions

Repeat the pipeline on the other two data sets; each has its own twist:

- `two_step_compulsivity` (lesson 9 model; 7 parameters): the parameter of interest is $w$; the two-step model's boundary estimates make priors or hierarchical fitting (lesson 8) attractive, and the one-trial-back regression (lesson 14) offers a model-agnostic cross-check.
- `reversal_anxiety` (lesson 12): the quantity of interest is learning-rate *adaptation*; compare the two-block RW analysis with a VKF fit (volatility learning rate $\lambda$) and ask which is more reliable.

## Summary

You have run a complete computational-psychiatry pipeline: model-agnostic description → candidate models → model recovery → model comparison → parameter recovery → symptom analysis with specificity, covariates, reliability and attenuation → posterior predictive check → write-up. Two lessons stand out: *the design limits what any comparison can detect* (calibrate the comparison with model recovery), and *estimation noise attenuates every correlation* (measure reliability and say so).

## References

- Daw, N. D. (2011). Trial-by-trial data analysis using computational models. In M. R. Delgado, E. A. Phelps & T. W. Robbins (Eds.), *Decision Making, Affect, and Learning: Attention and Performance XXIII* (pp. 3–38). Oxford University Press.
- Huys, Q. J. M., Pizzagalli, D. A., Bogdan, R., & Dayan, P. (2013). Mapping anhedonia onto reinforcement learning: A behavioural meta-analysis. *Biology of Mood & Anxiety Disorders, 3*, 12. https://doi.org/10.1186/2045-5380-3-12
- Kass, R. E., & Raftery, A. E. (1995). Bayes factors. *Journal of the American Statistical Association, 90*(430), 773–795. https://doi.org/10.1080/01621459.1995.10476572
- Lefebvre, G., Lebreton, M., Meyniel, F., Bourgeois-Gironde, S., & Palminteri, S. (2017). Behavioural and neural characterization of optimistic reinforcement learning. *Nature Human Behaviour, 1*, 0067. https://doi.org/10.1038/s41562-017-0067
- Palminteri, S., Wyart, V., & Koechlin, E. (2017). The importance of falsification in computational cognitive modeling. *Trends in Cognitive Sciences, 21*(6), 425–433. https://doi.org/10.1016/j.tics.2017.03.011
- Rigoux, L., Stephan, K. E., Friston, K. J., & Daunizeau, J. (2014). Bayesian model selection for group studies — revisited. *NeuroImage, 84*, 971–985. https://doi.org/10.1016/j.neuroimage.2013.08.065
- Stephan, K. E., Penny, W. D., Daunizeau, J., Moran, R. J., & Friston, K. J. (2009). Bayesian model selection for group studies. *NeuroImage, 46*(4), 1004–1017. https://doi.org/10.1016/j.neuroimage.2009.03.025
- Wilson, R. C., & Collins, A. G. E. (2019). Ten simple rules for the computational modeling of behavioral data. *eLife, 8*, e49547. https://doi.org/10.7554/eLife.49547
- Zorowitz, S., & Niv, Y. (2023). Improving the reliability of cognitive task measures: A narrative review. *Biological Psychiatry: Cognitive Neuroscience and Neuroimaging, 8*(8), 789–797. https://doi.org/10.1016/j.bpsc.2023.02.004